This notebook does not run in production. 

It was used to fine-tune the Pydantic classes when integrating data source 2, exporting the generated JSONs to a file and then comapring them with the ground truth of the dataset in order to validate. 

Nevertheless I decided to add this to the GitHub repo as a showcase of the model preparation process.

In [ ]:
%uv pip install docling
%uv pip install qwen-vl-utils
%uv pip install pymupdf

In [ ]:
from PIL import Image
import io
from IPython.display import display, JSON
from pydantic import BaseModel, Field, AliasChoices
from rich import print
import polars as pl
from typing import Optional
import json
from docling.datamodel.document import DocumentStream
from docling.datamodel.base_models import InputFormat
from docling.document_extractor import (
    DocumentExtractor, ExtractionFormatOption,
    ExtractionVlmPipeline, ImageDocumentBackend,
    PyPdfiumDocumentBackend, InputFormat
)
from docling.document_converter import PdfFormatOption, ImageFormatOption, DocumentConverter
import torch
import gc
import tempfile, os
from pathlib import Path
import fitz

In [ ]:
#Clearing cache of graphics card
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

In [ ]:
#Enabling GPU acceleration for faster model inference

from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice
from docling.datamodel.pipeline_options import VlmExtractionPipelineOptions
from docling.document_extractor import DocumentExtractor

acc_options = AcceleratorOptions(device=AcceleratorDevice.CUDA)
pipeline_options = VlmExtractionPipelineOptions(accelerator_options=acc_options)

In [ ]:
# This needs to be de-allocated at the end to avoid memory leaks (because it's VRAM and not regular RAM), although it's not absolutely necessary if it runs on Modal, it's a good safety guard.
extractor = DocumentExtractor(
    allowed_formats=[InputFormat.IMAGE],
    extraction_format_options={
        InputFormat.IMAGE: ExtractionFormatOption(
            pipeline_cls=ExtractionVlmPipeline,
            pipeline_options=pipeline_options,
            backend=ImageDocumentBackend,
        )
    }
)

In [ ]:
class InvoiceHeader_DS2(BaseModel):
    invoice_date: Optional[str] = Field(
        default=None, 
        description="The invoice or order date",
        examples=["Date", "Date:"],
        validation_alias=AliasChoices("Date", "Order date")
    )
    client: Optional[str] = Field(
        default=None, 
        description="The billed client or customer name",
        examples=["Bill To:", "Bill To:"],
        validation_alias=AliasChoices("Bill To:", "Bill To", "Customer Name", "Buyer")
    )
    client_address: Optional[str] = Field(
        default=None, 
        description="The shipping or delivery address",
        examples=["Ship To:", "Ship To"],
        validation_alias=AliasChoices("Ship To:", "Address", "Ship To")
    )

In [ ]:
class InvoiceItem_DS2(BaseModel):
    item_desc: Optional[str] = Field(
        default=None, 
        description="The item or product description",
        examples=["Item"],
        validation_alias=AliasChoices("Item", "Item description", "Item name", "Product name")
    )
    item_qty: Optional[str] = Field(
        default=None, 
        description="The quantity ordered for this product",
        examples=["Quantity"],
        validation_alias=AliasChoices("Quantity", "Item quantity")
    )
    item_rate: Optional[str] = Field(
        default=None,
        description="The unit price or rate",
        examples=["Rate"],
        validation_alias=AliasChoices("Rate", "Price", "Item net price", "Unit cost")
    )

In [ ]:
class InvoiceSummary_DS2(BaseModel):
    order_id: Optional[str] = Field(
        default=None,
        description="The order or invoice ID/number",
        examples=["Order ID:", "Order ID"],
        validation_alias=AliasChoices("Order ID", "order")
    )
    discount: Optional[str] = Field(
        default=None, 
        description="Discount amount applied",
        examples=["Discount"],
        validation_alias=AliasChoices("Discount")
    )
    shipping: Optional[str] = Field(
        default=None, 
        description="Shipping fee",
        examples=["Shipping:", "Shipping"],
        validation_alias=AliasChoices("Shipping", "Shipping fee")
    )

In [ ]:
class InvoiceDocument_DS2(BaseModel):
    header: InvoiceHeader_DS2
    items: list[InvoiceItem_DS2]
    summary: InvoiceSummary_DS2

In [ ]:
def pdf_to_image_bytes_list(pdf_bytes: bytes, dpi: int = 150) -> list[bytes]:
    # Convert each page of a PDF to JPEG bytes.
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    pages_as_images = []
    for page in doc:
        mat = fitz.Matrix(dpi / 72, dpi / 72)  # 72 is PDF's base DPI
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        pages_as_images.append(pix.tobytes("jpeg"))
    doc.close()
    return pages_as_images

In [ ]:
def img_bytes_to_json(img_bytes: bytes, pydantic_model):
    # Write to a temp file and delete right after to make it easier for the extractor to parse
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f:
        f.write(img_bytes)
        tmp_path = Path(f.name)
    try:
        result = extractor.extract(
            source=tmp_path,
            template=pydantic_model,   #The pydantic model that defines the expected JSON structure
            raises_on_error=True,
        )
    finally:
        os.unlink(tmp_path)

    raw = result.pages[0]
    print("extracted_data type:", type(raw.extracted_data))
    print("extracted_data:", raw.extracted_data)
    print("raw_text:", raw.raw_text)

    return json.loads(result.json())["pages"][0]["extracted_data"]

In [ ]:
fileList = os.listdir('/root')
pdfPaths = [f'/root/{f}' for f in fileList if f.endswith('.pdf')]
pdfPaths.sort()

In [ ]:
print(pdfPaths)

In [ ]:
# with open("/root/invoice_Maria Zettner_24429.pdf", "rb") as f:
#     pdf_file_1 = f.read()

# pdf_pages_file_1 = pdf_to_image_bytes_list(pdf_file_1)
# img_bytes = pdf_pages_file_1[0]

In [ ]:
result = img_bytes_to_json(img_bytes, InvoiceDocument_DS2)

In [ ]:
print(result)

In [ ]:
def prettyfy_extraction_result(result):
    return json.dumps(json.loads(result.json())['pages'][0]['extracted_data'], indent=2)

In [ ]:
def parse_invoice_bytes(img_bytes: bytes):
    # Save bytes to temp file with correct extension
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f:
        f.write(img_bytes)
        tmp_path = Path(f.name)

    try:
        result = extractor.extract(
            source=tmp_path,
            template=InvoiceDocument_DS2,
            raises_on_error=True,
        )
        #print(result)
    finally:
        os.unlink(tmp_path)  # clean up

    return prettyfy_extraction_result(result)

In [ ]:
#result = parse_invoice_bytes(img_bytes)

Here we dump the first 70 invoices into a .txt file in order to be compared against the ground truth, to asses the accuracy of the model

In [ ]:
with open("first_70_invoices.txt", "w+") as f:
    f.write("[")
    for i in range(70):
        curr_pdf_path = pdfPaths[i]
        with open(curr_pdf_path, "rb") as f_pdf:
            curr_pdf_file = f_pdf.read()
        pdf_pages_file_1 = pdf_to_image_bytes_list(curr_pdf_file)
        img_bytes = pdf_pages_file_1[0]
        
        curr_result = parse_invoice_bytes(img_bytes)
        f.write("//" + curr_pdf_path + ":\n" + curr_result + "," + "\n")
        print(f"Processed invoice {i+1}/70")
    f.write("]")

Yeah, we have to do manual garbage collection in Python, like in the good old days of C/C++ in college, because this is VRAM and not regular RAM.

In [ ]:
# memory de-allocation
del extractor

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")